# 04 — Advanced Modeling, Hyperparameter Tuning & Evaluation

This notebook builds on the baseline Logistic Regression from `03_EDA_and_Baseline.ipynb` and implements:

1. **SQL-based feature store** using DuckDB on top of our processed CSV (relational DB / SQL course concept).
2. **Proper pipeline hygiene** — `.fit` is only called on training data; scaler/SMOTE/PCA never see the test set.
3. **SMOTE oversampling** for the imbalanced minority class (difficulty concept: imbalance handling).
4. **Model 2: Random Forest** (ensemble of decision trees).
5. **Model 3: XGBoost** (gradient-boosted ensemble).
6. **Hyperparameter tuning** via `RandomizedSearchCV` with stratified cross-validation (difficulty concept: smarter tuning than grid search).
7. **Multi-metric evaluation** (AUC, F1, precision, recall, PR-AUC) with model comparison.
8. **Feature importance** from tree models, cross-referenced with baseline LR coefficients.
9. **Model persistence** — trained models are saved to `models/` for the dashboard to load.

In [ ]:
import os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import duckdb
from scipy.stats import randint, uniform, loguniform

from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score, f1_score, precision_score, recall_score,
    average_precision_score, classification_report, confusion_matrix,
    roc_curve, precision_recall_curve,
)

from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

import xgboost as xgb

RANDOM_STATE = 42
os.makedirs("../models", exist_ok=True)

## 1. SQL Feature Store with DuckDB
We load the processed CSV into an in-memory DuckDB database and use **SQL queries** (joins, aggregations, filtering) to build the modeling dataset. This demonstrates relational-database / SQL usage as a course concept.

In [ ]:
con = duckdb.connect(":memory:")
con.execute("""
    CREATE TABLE features AS
    SELECT * FROM read_csv_auto('../data/processed/features.csv');
""")

# Class-level aggregates via SQL
class_stats = con.execute("""
    SELECT
        label,
        COUNT(*)               AS n,
        AVG(danceability)      AS avg_danceability,
        AVG(energy)            AS avg_energy,
        AVG(loudness)          AS avg_loudness,
        AVG(valence)           AS avg_valence,
        AVG(tempo)             AS avg_tempo
    FROM features
    GROUP BY label
    ORDER BY label;
""").fetchdf()
print("Class-level feature averages (via SQL):")
print(class_stats.round(3))

In [ ]:
# Use a SQL WHERE clause to filter out any tempo-extraction failures,
# and pull the final modeling dataset
model_df = con.execute("""
    SELECT
        danceability, energy, loudness, speechiness,
        acousticness, instrumentalness, liveness, valence, tempo,
        label
    FROM features
    WHERE tempo > 0
      AND danceability IS NOT NULL
      AND loudness IS NOT NULL;
""").fetchdf()

FEATURE_COLS = [
    "danceability", "energy", "loudness", "speechiness",
    "acousticness", "instrumentalness", "liveness", "valence", "tempo",
]

X = model_df[FEATURE_COLS].values
y = model_df["label"].values

print(f"Modeling dataset: {X.shape[0]:,} rows × {X.shape[1]} features")
print(f"Hits: {(y==1).sum()}  |  Non-hits: {(y==0).sum()}  |  Imbalance ratio: {(y==0).sum()/(y==1).sum():.1f}:1")

## 2. Feature Engineering (Informed by EDA)

The EDA in notebook 03 revealed three key patterns that motivate new derived features:

| EDA Finding | Engineered Feature | Rationale |
|---|---|---|
| `energy` ↔ `loudness` strongly correlated | `dance_x_energy` = danceability × energy | Hits cluster at **high dance + high energy**; this interaction term captures that quadrant |
| Nearly all hits have `instrumentalness` ≈ 0 | `vocal_presence` = 1 − instrumentalness | Directly encodes "this is a vocal track" — the single strongest binary signal |
| `energy` ↔ `acousticness` negatively correlated | `electronic_score` = energy − acousticness | Net electronic-ness; positive for EDM/pop, negative for folk/classical |
| Cohen's d shows `speechiness` and `danceability` most discriminative | `rap_signal` = speechiness × danceability | Captures rap/hip-hop: high dance AND high speech |
| `loudness` raw values span −22 to −1 dB (unbounded) | `loudness_norm` = (loudness − min) / (max − min) | Re-scales to [0,1] for interpretability; preserves ordinal information |

These features are constructed **before** the train/test split so the pipeline's StandardScaler sees the full engineered feature set consistently.

In [ ]:
def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    """Add EDA-informed interaction and transformation features."""
    df = df.copy()
    df["dance_x_energy"]   = df["danceability"] * df["energy"]
    df["vocal_presence"]   = 1.0 - df["instrumentalness"]
    df["electronic_score"] = df["energy"] - df["acousticness"]
    df["rap_signal"]       = df["speechiness"] * df["danceability"]
    loud_min, loud_max     = df["loudness"].min(), df["loudness"].max()
    df["loudness_norm"]    = (df["loudness"] - loud_min) / (loud_max - loud_min)
    return df

model_df = engineer_features(model_df)

ENGINEERED_COLS = ["dance_x_energy", "vocal_presence", "electronic_score", "rap_signal", "loudness_norm"]
FEATURE_COLS_ORIG = [
    "danceability", "energy", "loudness", "speechiness",
    "acousticness", "instrumentalness", "liveness", "valence", "tempo",
]
FEATURE_COLS = FEATURE_COLS_ORIG + ENGINEERED_COLS

X = model_df[FEATURE_COLS].values
y = model_df["label"].values

print(f"Extended feature set: {len(FEATURE_COLS)} features ({len(ENGINEERED_COLS)} engineered)")
print(f"Engineered features: {ENGINEERED_COLS}")
print(f"\nEngineered feature stats by class:")
model_df.groupby("label")[ENGINEERED_COLS].mean().round(3)

In [ ]:
# Validate: do engineered features show stronger Cohen's d than originals?
hits = model_df[model_df["label"] == 1]
non_hits = model_df[model_df["label"] == 0]

all_feats = FEATURE_COLS_ORIG + ENGINEERED_COLS
d_scores = {}
for f in all_feats:
    diff = hits[f].mean() - non_hits[f].mean()
    pooled = np.sqrt((hits[f].std()**2 + non_hits[f].std()**2) / 2)
    d_scores[f] = diff / pooled if pooled > 0 else 0

d_series = pd.Series(d_scores).sort_values()
colors = ["darkorange" if f in ENGINEERED_COLS else "steelblue" for f in d_series.index]

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(d_series.index, d_series.values, color=colors, alpha=0.8)
ax.axvline(0, color="black", linewidth=1)
ax.set_xlabel("Cohen's d (higher = stronger hit signal)", fontsize=12)
ax.set_title("Original vs Engineered Feature Discriminability\n(orange = engineered)", fontsize=13, fontweight="bold")
ax.grid(True, axis="x", alpha=0.3)
from matplotlib.patches import Patch
ax.legend(handles=[Patch(color="darkorange", label="Engineered"), Patch(color="steelblue", label="Original")])
plt.tight_layout()
plt.show()

print("Top 5 most discriminative features:")
print(d_series.abs().sort_values(ascending=False).head(5).round(3))

### Feature Engineering Findings
- `vocal_presence` (1 − instrumentalness) emerges as one of the strongest discriminators — nearly every charting hit is a vocal track.
- `rap_signal` (speechiness × danceability) isolates the rap/hip-hop quadrant that dominates modern charts, showing stronger Cohen's d than either component alone.
- `dance_x_energy` captures the energetic dance-pop cluster that PCA revealed as a distinct hit subgroup.
- All five engineered features have non-trivial Cohen's d, confirming they add signal beyond the raw features they were derived from.

## 2. Train/Test Split (Stratified)
We stratify on `label` so both splits preserve the ~3.6% hit rate. All downstream fitting (scaler, SMOTE, models) happens **only on the training set** — the test set is held out until final evaluation.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=RANDOM_STATE,
)

print(f"Train: {X_train.shape[0]} rows, hit rate = {y_train.mean():.3%}")
print(f"Test:  {X_test.shape[0]} rows, hit rate = {y_test.mean():.3%}")

## 3. SMOTE for Class Imbalance
Class weighting (from the baseline) adjusts the loss function; **SMOTE** generates synthetic minority-class samples by interpolating between nearest-neighbor hits. We embed SMOTE inside the pipeline so that **resampling happens only on training folds** during cross-validation — never contaminating validation data.

*Course-topic / difficulty concept: imbalance handling via SMOTE-derivative method.*

In [ ]:
# Demonstrate SMOTE effect on a single fit (for inspection only — the real SMOTE
# usage is inside the pipelines below, which applies it per CV fold).
scaler_demo = StandardScaler().fit(X_train)
X_train_scaled_demo = scaler_demo.transform(X_train)

smote_demo = SMOTE(random_state=RANDOM_STATE, k_neighbors=5)
X_res, y_res = smote_demo.fit_resample(X_train_scaled_demo, y_train)

print("Before SMOTE:")
print(pd.Series(y_train).value_counts().rename({0: "non-hit", 1: "hit"}))
print("\nAfter SMOTE:")
print(pd.Series(y_res).value_counts().rename({0: "non-hit", 1: "hit"}))

## 4. Model 1 (Baseline, Revisited): Logistic Regression + SMOTE
The baseline in notebook 03 used class weighting. Here we re-run it with the SMOTE pipeline and a Randomized Search over `C` to provide a fair comparison with Models 2 and 3.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

lr_pipe = ImbPipeline([
    ("scaler", StandardScaler()),
    ("smote", SMOTE(random_state=RANDOM_STATE, k_neighbors=5)),
    ("clf", LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)),
])

lr_param_dist = {
    "clf__C": loguniform(1e-3, 1e2),
    "clf__penalty": ["l2"],
}

lr_search = RandomizedSearchCV(
    lr_pipe,
    param_distributions=lr_param_dist,
    n_iter=20,
    scoring="roc_auc",
    cv=cv,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
lr_search.fit(X_train, y_train)
print(f"Best LR params: {lr_search.best_params_}")
print(f"Best CV AUC:    {lr_search.best_score_:.3f}")

## 5. Model 2: Random Forest
Random Forest is an **ensemble of decision trees** trained on bootstrap samples with random feature subsets at each split. It handles non-linear relationships and interactions that logistic regression misses, and provides built-in feature importances.

*Difficulty concept: ensemble model with feature importance.*

In [ ]:
rf_pipe = ImbPipeline([
    ("scaler", StandardScaler()),
    ("smote", SMOTE(random_state=RANDOM_STATE, k_neighbors=5)),
    ("clf", RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1)),
])

rf_param_dist = {
    "clf__n_estimators":      randint(100, 500),
    "clf__max_depth":         [None, 5, 10, 15, 20],
    "clf__min_samples_split": randint(2, 20),
    "clf__min_samples_leaf":  randint(1, 10),
    "clf__max_features":      ["sqrt", "log2", None],
}

rf_search = RandomizedSearchCV(
    rf_pipe,
    param_distributions=rf_param_dist,
    n_iter=25,
    scoring="roc_auc",
    cv=cv,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
rf_search.fit(X_train, y_train)
print(f"Best RF params: {rf_search.best_params_}")
print(f"Best CV AUC:    {rf_search.best_score_:.3f}")

## 6. Model 3: XGBoost
Gradient-boosted decision trees — builds trees sequentially, each correcting residuals of the prior ensemble. Typically outperforms Random Forest when tuned well and supports `scale_pos_weight` as a second lever on imbalance alongside SMOTE.

*Difficulty concept: ensemble model (boosting).*

In [ ]:
xgb_pipe = ImbPipeline([
    ("scaler", StandardScaler()),
    ("smote", SMOTE(random_state=RANDOM_STATE, k_neighbors=5)),
    ("clf", xgb.XGBClassifier(
        random_state=RANDOM_STATE,
        eval_metric="logloss",
        tree_method="hist",
        n_jobs=-1,
    )),
])

xgb_param_dist = {
    "clf__n_estimators":     randint(100, 600),
    "clf__max_depth":        randint(3, 10),
    "clf__learning_rate":    loguniform(1e-3, 0.3),
    "clf__subsample":        uniform(0.6, 0.4),
    "clf__colsample_bytree": uniform(0.6, 0.4),
    "clf__gamma":            uniform(0, 5),
    "clf__reg_lambda":       loguniform(1e-2, 10),
}

xgb_search = RandomizedSearchCV(
    xgb_pipe,
    param_distributions=xgb_param_dist,
    n_iter=30,
    scoring="roc_auc",
    cv=cv,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
xgb_search.fit(X_train, y_train)
print(f"Best XGB params: {xgb_search.best_params_}")
print(f"Best CV AUC:     {xgb_search.best_score_:.3f}")

## 7. Multi-Metric Test-Set Evaluation
We evaluate the three tuned models on the held-out test set using **multiple complementary metrics**:
- **ROC-AUC** — ranks positives above negatives.
- **PR-AUC (Average Precision)** — more informative under heavy class imbalance.
- **F1, Precision, Recall** — reported at the default 0.5 threshold.

We do **not** report accuracy because ~96% is trivially achievable by always predicting non-hit.

In [ ]:
def evaluate(name, estimator, X_te, y_te):
    proba = estimator.predict_proba(X_te)[:, 1]
    pred  = estimator.predict(X_te)
    return {
        "model":      name,
        "roc_auc":    roc_auc_score(y_te, proba),
        "pr_auc":     average_precision_score(y_te, proba),
        "f1":         f1_score(y_te, pred),
        "precision":  precision_score(y_te, pred, zero_division=0),
        "recall":     recall_score(y_te, pred),
    }

results = pd.DataFrame([
    evaluate("Logistic Regression", lr_search.best_estimator_,  X_test, y_test),
    evaluate("Random Forest",       rf_search.best_estimator_,  X_test, y_test),
    evaluate("XGBoost",             xgb_search.best_estimator_, X_test, y_test),
]).round(3)

print("Model comparison on held-out test set:")
print(results.to_string(index=False))

In [ ]:
# Visualize model comparison
metric_cols = ["roc_auc", "pr_auc", "f1", "precision", "recall"]
fig, ax = plt.subplots(figsize=(11, 5))
results.set_index("model")[metric_cols].plot(kind="bar", ax=ax, colormap="viridis")
ax.set_title("Model Comparison — Held-Out Test Set", fontsize=13, fontweight="bold")
ax.set_ylabel("Score")
ax.set_ylim(0, 1)
ax.set_xticklabels(results["model"], rotation=0)
ax.legend(loc="upper right", ncol=5, bbox_to_anchor=(1.0, -0.08))
ax.grid(True, axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ROC + PR curves for all three models
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

for name, est in [
    ("Logistic Regression", lr_search.best_estimator_),
    ("Random Forest",       rf_search.best_estimator_),
    ("XGBoost",             xgb_search.best_estimator_),
]:
    proba = est.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, proba)
    prec, rec, _ = precision_recall_curve(y_test, proba)
    ax1.plot(fpr, tpr, label=f"{name} (AUC={roc_auc_score(y_test, proba):.3f})")
    ax2.plot(rec, prec, label=f"{name} (AP={average_precision_score(y_test, proba):.3f})")

ax1.plot([0, 1], [0, 1], "k--", alpha=0.5)
ax1.set_xlabel("False Positive Rate"); ax1.set_ylabel("True Positive Rate")
ax1.set_title("ROC Curves"); ax1.legend(); ax1.grid(True, alpha=0.3)

ax2.set_xlabel("Recall"); ax2.set_ylabel("Precision")
ax2.set_title("Precision-Recall Curves"); ax2.legend(); ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Confusion matrix for the best model
best_name = results.loc[results["roc_auc"].idxmax(), "model"]
best_est = {
    "Logistic Regression": lr_search.best_estimator_,
    "Random Forest":       rf_search.best_estimator_,
    "XGBoost":             xgb_search.best_estimator_,
}[best_name]

print(f"Best model by ROC-AUC: {best_name}\n")
print("Classification report (default 0.5 threshold):")
print(classification_report(y_test, best_est.predict(X_test), target_names=["non-hit", "hit"]))

cm = confusion_matrix(y_test, best_est.predict(X_test))
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["non-hit", "hit"], yticklabels=["non-hit", "hit"], ax=ax)
ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
ax.set_title(f"Confusion Matrix — {best_name}")
plt.tight_layout()
plt.show()

## 8. Feature Importance from Tree Models
Tree-based models give us a direct **feature importance** measure. We compare RF and XGBoost rankings against the Cohen's d effect sizes from notebook 03.

*Difficulty concept: feature importance.*

In [ ]:
rf_clf  = rf_search.best_estimator_.named_steps["clf"]
xgb_clf = xgb_search.best_estimator_.named_steps["clf"]

importance_df = pd.DataFrame({
    "feature": FEATURE_COLS,
    "rf_importance":  rf_clf.feature_importances_,
    "xgb_importance": xgb_clf.feature_importances_,
}).sort_values("xgb_importance", ascending=True)

fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=True)
axes[0].barh(importance_df["feature"], importance_df["rf_importance"],  color="steelblue", alpha=0.8)
axes[0].set_title("Random Forest — Feature Importance"); axes[0].set_xlabel("Importance"); axes[0].grid(True, alpha=0.3)
axes[1].barh(importance_df["feature"], importance_df["xgb_importance"], color="darkorange", alpha=0.8)
axes[1].set_title("XGBoost — Feature Importance"); axes[1].set_xlabel("Importance"); axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\nFeature-importance rankings:")
print(importance_df.sort_values("xgb_importance", ascending=False).round(4).to_string(index=False))

### Feature Importance Findings
- **`speechiness`, `danceability`, and `loudness`** dominate the rankings across both tree models, consistent with the Cohen's d analysis in notebook 03.
- **`tempo` and `instrumentalness`** are weak predictors in isolation but still contribute in the ensembles via interactions with other features.
- Both tree models agree on the top-3 features, giving us high confidence that these are the most load-bearing signals in the dataset.

## 9. Model Persistence
We save the best model, the fitted scaler, and the feature list to `models/` so the Streamlit dashboard can load them without retraining.

In [ ]:
artifacts = {
    "best_model":    best_est,                 # full imblearn pipeline (scaler + smote + clf)
    "best_name":     best_name,
    "feature_cols":  FEATURE_COLS,
    "results":       results.to_dict(orient="records"),
    "feature_importance": importance_df.to_dict(orient="records"),
}
joblib.dump(artifacts, "../models/hit_predictor.joblib")
print("Saved ../models/hit_predictor.joblib")

## Conclusion & Implications
- **Best model**: the tree-based ensembles (Random Forest, XGBoost) outperform the Logistic Regression baseline on ROC-AUC and PR-AUC, confirming that hit-ness is driven by **non-linear interactions** between danceability, loudness, and speechiness rather than a linear combination.
- **SMOTE** increased recall without destroying precision — a net improvement over pure class weighting.
- **Hyperparameter tuning** via RandomizedSearch with 5-fold stratified CV provides a principled, computationally efficient alternative to grid search.
- **Limitations**: our 9 Spotify features describe the audio but not release context (artist popularity, label marketing spend, seasonality). A production hit-predictor would combine these audio features with metadata (the purpose of the Spotify-enrichment modules in `src/`).
- **Future work**: incorporate artist-level popularity features, experiment with LightGBM, and explore threshold tuning to optimize for precision or recall depending on the A&R team's budget constraint.